### Datensatz Photolab Archives Albums
Import und öffnen des Browsers

In [52]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
import re
import time
import os
from bs4 import BeautifulSoup
from bs4 import NavigableString
from urllib.parse import urljoin, urlsplit, urlunsplit


# Start driver
driver = webdriver.Firefox()

# Open list page
driver.get("https://cds.cern.ch/collection/PhotoLab%20Archives?ln=en")

# Give page time to load
time.sleep(2)

### 100 Ergebnisse

In [53]:
# 2. Click the "[>> mehr]" link
mehr_link = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "[>> more]"))
)
mehr_link.click()

# 3. Select "100 Ergebnisse" in the <select name="rg">
select_element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.NAME, "rg"))
)
select = Select(select_element)
select.select_by_value("100")   # value="100"

# 4. Click the "Suchen" submit button
search_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.NAME, "action_search"))
)
search_button.click()

print("Navigation complete. Page refreshed with 100 results.")

Navigation complete. Page refreshed with 100 results.


### Find the dates of the records
table Verschachtelung, td

In [54]:
soup = BeautifulSoup(driver.page_source, "html.parser")

# Get table rows again after navigation
results = [
    tr for tr in soup.select("tbody > tr")
    if len(tr.find_all("td")) == 2
]

print("Found rows:", len(results))


date_pattern = re.compile(
    r"(?i)(\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4})|"  # e.g., 27 Sep 1961
    r"([A-Za-z]{3,9}\s+\d{4})|"                # e.g., Sep 1977
    r"(\d{4})|"                                # e.g., 1961
    r"(No date)"                               # literal "No date"
)

dates = []


for r in results:
    td = r.find_all("td")[1]
    brs = td.find_all("br")

    date_text = None

    for br in brs:
        sib = br.next_sibling

        # Skip empties/tags
        while sib and (not isinstance(sib, NavigableString) or not sib.strip()):
            sib = sib.next_sibling
        if not sib:
            continue

        text = sib.strip()

        # Ignore description lines containing [...]
        if "[...]" in text:
            continue

        # Extract if it matches a real date
        match = date_pattern.search(text)
        if match:
            date_text = match.group(0)
            break

    dates.append(date_text)
    print("DATE:", date_text)

Found rows: 100
DATE: 27 Sep 1961
DATE: 25 Mar 1971
DATE: 1971
DATE: Dec 1977
DATE: Sep 1977
DATE: 9 Oct 2000
DATE: 9 Oct 2000
DATE: 9 Oct 2000
DATE: Jun 1986
DATE: Aug 1986
DATE: Jul 1986
DATE: Aug 1988
DATE: No date
DATE: 13 Jan 1997
DATE: 2 May 1995
DATE: 18 May 1995
DATE: the 1986
DATE: Feb 2000
DATE: Feb 2000
DATE: Mar 1976
DATE: Jul 1988
DATE: May 1988
DATE: May 1988
DATE: Apr 1988
DATE: Jul 1988
DATE: IBM 3090
DATE: Aug 1987
DATE: Nov 1987
DATE: 2 Apr 1994
DATE: Jan 1994
DATE: Mar 1992
DATE: Dec 1992
DATE: Dec 1992
DATE: 14 Apr 1994
DATE: Apr 1993
DATE: Jun 1993
DATE: Jan 1999
DATE: Aug 1990
DATE: Aug 1990
DATE: Feb 2004
DATE: 4 Jun 2003
DATE: Oct 1996
DATE: Mar 1996
DATE: Sep 1992
DATE: 12 Jun 1991
DATE: Apr 1991
DATE: Jan 1991
DATE: Jul 1996
DATE: 1960
DATE: Sep 1993
DATE: Sep 1993
DATE: None
DATE: 30 Sep 1992
DATE: 15 May 1985
DATE: Nov 1985
DATE: Dec 1985
DATE: Dec 1985
DATE: 1981
DATE: Jan 1977
DATE: Jan 1977
DATE: May 1975
DATE: Feb 1980
DATE: 6 Feb 1974
DATE: May 1975
DAT

### Add row number column

In [55]:
# MODULE: add row numbers only for pictures
def add_picture_row_numbers(results):
    """
    Returns a list of row numbers (1-based) for rows that contain pictures.
    Skips rows without images, so it aligns with your image list.
    """
    row_numbers = []
    for i, r in enumerate(results, start=1):
        img_td = r.find_all("td")[0]  # left column for image
        img = img_td.find("img")
        if img and img.get("src"):
            row_numbers.append(i)
    return row_numbers

# BUILD row number list  ← ← ← THIS LINE WAS MISSING
row_numbers = add_picture_row_numbers(results)


### Bilder Anzahl
Tirage

In [56]:
tirages = []

for r in results:
    td = r.find_all("td")[1]

    tirage_text = None

    # Find the <em> tag with the word "Tirage"
    em_tag = td.find("em", string=lambda s: s and "Tirage" in s)

    if em_tag:
        # The number is usually after the em tag in the same <br> text node
        sib = em_tag.next_sibling

        # Skip to next meaningful text
        while sib and (not isinstance(sib, NavigableString) or not sib.strip()):
            sib = sib.next_sibling

        if sib:
            tirage_text = sib.strip()     # Usually "45"
    
    tirages.append(tirage_text)
    print("TIRAGE:", tirage_text)


TIRAGE: : 45
TIRAGE: : 16
TIRAGE: : 12
TIRAGE: : 12
TIRAGE: : 4
TIRAGE: : 13
TIRAGE: : 32
TIRAGE: : 53
TIRAGE: : 4
TIRAGE: : 11
TIRAGE: : 7
TIRAGE: : 10
TIRAGE: : 1
TIRAGE: : 1
TIRAGE: : 14
TIRAGE: : 3
TIRAGE: None
TIRAGE: : 12
TIRAGE: : 2
TIRAGE: : 2
TIRAGE: : 11
TIRAGE: : 3
TIRAGE: : 6
TIRAGE: : 4
TIRAGE: : 44
TIRAGE: : 1
TIRAGE: : 12
TIRAGE: : 3
TIRAGE: : 8
TIRAGE: : 28
TIRAGE: : 14
TIRAGE: : 8
TIRAGE: : 59
TIRAGE: : 29
TIRAGE: : 11
TIRAGE: : 21
TIRAGE: : 6
TIRAGE: : 1
TIRAGE: : 2
TIRAGE: : 2
TIRAGE: : 1
TIRAGE: : 8
TIRAGE: : 2
TIRAGE: : 3
TIRAGE: : 15
TIRAGE: : 3
TIRAGE: : 17
TIRAGE: : 2
TIRAGE: : 4
TIRAGE: : 5
TIRAGE: : 3
TIRAGE: None
TIRAGE: : 13
TIRAGE: : 6
TIRAGE: : 2
TIRAGE: : 2
TIRAGE: : 6
TIRAGE: : 13
TIRAGE: : 61
TIRAGE: : 32
TIRAGE: : 2
TIRAGE: : 18
TIRAGE: : 2
TIRAGE: : 4
TIRAGE: : 1
TIRAGE: : 4
TIRAGE: : 3
TIRAGE: : 3
TIRAGE: : 1
TIRAGE: : 1
TIRAGE: : 4
TIRAGE: : 7
TIRAGE: : 5
TIRAGE: : 1
TIRAGE: : 2
TIRAGE: : 30
TIRAGE: : 2
TIRAGE: : 9
TIRAGE: : 6
TIRAGE: : 2
TIRAGE: : 

### Find the titles, URLs of the records, URLs of the Bilder
class='titlelink' und a href=.....

In [57]:
soup = BeautifulSoup(driver.page_source, "html.parser")

titles = []
links = []          # album / record links
image_links = []    # NEW: direct image links

results = [
    tr for tr in soup.select("tbody > tr")
    if len(tr.find_all("td")) == 2
]

base_url = "https://cds.cern.ch"

for r in results:
    cells = r.find_all("td")

    # -------------------------
    # TEXT / ALBUM LINK COLUMN
    # -------------------------
    text_td = cells[1]
    a = text_td.find("a", class_="titlelink")

    if a:
        titles.append(a.get_text(strip=True))
        links.append(a.get("href"))
    else:
        titles.append(None)
        links.append(None)

    # -------------------------
    # IMAGE LINK COLUMN (NEW)
    # -------------------------
    image_td = cells[0]
    img = image_td.find("img")

    if img and img.get("src"):
        img_src = img.get("src")

        # make absolute URL
        full_img_url = urljoin(base_url, img_src)

        # OPTIONAL: upgrade thumbnail → full image if needed
        # CDS usually uses /files/XXXX.jpg → this is already the real file
        image_links.append(full_img_url)
    else:
        image_links.append(None)

# -------------------------
# FINAL URL NORMALIZATION
# -------------------------
record_urls = [urljoin(base_url, href) if href else None for href in links]

# -------------------------
# DEBUGGING
# -------------------------
print("Titles:", len(titles))
print("Record URLs:", len(record_urls))
print("Image URLs:", image_links)

missing_images = [i for i, url in enumerate(image_links) if url is None]
print("Missing image indices:", missing_images)


Titles: 100
Record URLs: 100
Image URLs: ['https://cds.cern.ch/record/1800778/files/4465.jpg?subformat=icon-180', 'https://cds.cern.ch/record/2474015/files/icon640-SPECIAL_X_CERN_00192_0587.jpg?subformat=icon-640', 'https://cds.cern.ch/record/2474073/files/icon180-SPECIAL_X_CERN_00192_0645.jpg?subformat=icon-180', 'https://cds.cern.ch/record/2473694/files/icon180-SPECIAL_X_CERN_00192_0266.jpg?subformat=icon-180', 'https://cds.cern.ch/record/2473706/files/icon180-SPECIAL_X_CERN_00192_0278.jpg?subformat=icon-180', 'https://cds.cern.ch/record/2330196/files/2000-10-290X_01.jpg?subformat=icon-180', 'https://cds.cern.ch/record/2330166/files/2000-10-283X_01.jpg?subformat=icon-180', 'https://cds.cern.ch/record/2330232/files/2000-10-291X_01.jpg?subformat=icon-180', 'https://cds.cern.ch/record/2412275/files/icon1440-1986-06_X_CERN_00084_0240.jpg?subformat=icon-1440', 'https://cds.cern.ch/record/2412774/files/icon640-1986-07_X_CERN_00088_0025.jpg?subformat=icon-640', 'https://cds.cern.ch/record/2

### File name
take the src and cut the link so that it is only the file name

In [58]:
image_filenames = []

for r in results:
    tds = r.find_all("td")
    left_td = tds[0]  # the image column

    first_img = left_td.find("img")
    if not first_img:
        image_filenames.append(None)
        continue

    url = first_img.get("src", "")
    
    # Extract filename: part after /files/
    if "/files/" in url:
        file_part = url.split("/files/")[1]
        # Remove extension and query params
        base = file_part.split(".jpg")[0]
        filename = base.strip()
    else:
        filename = None

    image_filenames.append(filename)
    print("IMAGE FILENAME:", filename)


IMAGE FILENAME: 4465
IMAGE FILENAME: icon640-SPECIAL_X_CERN_00192_0587
IMAGE FILENAME: icon180-SPECIAL_X_CERN_00192_0645
IMAGE FILENAME: icon180-SPECIAL_X_CERN_00192_0266
IMAGE FILENAME: icon180-SPECIAL_X_CERN_00192_0278
IMAGE FILENAME: 2000-10-290X_01
IMAGE FILENAME: 2000-10-283X_01
IMAGE FILENAME: 2000-10-291X_01
IMAGE FILENAME: icon1440-1986-06_X_CERN_00084_0240
IMAGE FILENAME: icon640-1986-07_X_CERN_00088_0025
IMAGE FILENAME: icon640-1986-07_X_CERN_00086_0014
IMAGE FILENAME: icon180-1988-08_X_CERN_00003_0001
IMAGE FILENAME: Fidecaro-Team.gif?subformat=icon
IMAGE FILENAME: 1997-01-009X_01_icon180
IMAGE FILENAME: icon1440-1995-05-012_X_CERN_04315_0001
IMAGE FILENAME: icon640-1995-05-021_X_CERN_00315_0001
IMAGE FILENAME: Nambu
IMAGE FILENAME: 2000-02-019X_14_icon1440
IMAGE FILENAME: 2000-02-019X_01_icon180
IMAGE FILENAME: 76-3-046
IMAGE FILENAME: icon1440-1988-07_X_CERN_00009_0200
IMAGE FILENAME: icon1440-1988-05_X_CERN_00019_0105
IMAGE FILENAME: icon640-1988-05_X_CERN_00019_0103
IMAG

### Description text
Now with selenium click the record and take the description text

In [59]:
descriptions = []

for idx, record_url in enumerate(record_urls, start=1):
    print(f"\n>>> Opening record {idx}/{len(record_urls)}: {record_url}")

    try:
        driver.get(record_url)
    except Exception as e:
        print("   ERROR: Could not load page:", e)
        descriptions.append(None)
        continue

    time.sleep(1.2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Debug: check if the page loaded correctly
    if not soup.find("title"):
        print("   WARNING: Page title missing — page may not have loaded!")
        descriptions.append(None)
        continue

    # Debug: check if album-description exists
    desc_divs = soup.select("div.album-description")
    print("   Found description divs:", len(desc_divs))

    if len(desc_divs) == 0:
        print("   DEBUG: No album-description div found. Saving None.")
        descriptions.append(None)
        continue

    # Extract text
    desc_parts = []
    for div in desc_divs:
        for p in div.find_all("p"):
            text = p.get_text(strip=True)
            if text:
                desc_parts.append(text)

    full_description = " ".join(desc_parts) if desc_parts else None
    descriptions.append(full_description)

    print("   DESCRIPTION:", full_description)



>>> Opening record 1/100: https://cds.cern.ch/record/2945022?ln=en
   Found description divs: 2
   DESCRIPTION: None

>>> Opening record 2/100: https://cds.cern.ch/record/2933265?ln=en
   Found description divs: 2
   DESCRIPTION: Concert avec des oeuvres de H. Vecchi, T. Morley,  J. Maudit, F. Poulenc, B. Britten, C. Ph. E. Bach, G. Fauré, etc.

>>> Opening record 3/100: https://cds.cern.ch/record/2933125?ln=en
   Found description divs: 2
   DESCRIPTION: None

>>> Opening record 4/100: https://cds.cern.ch/record/2932507?ln=en
   Found description divs: 2
   DESCRIPTION: CERN's contributions to modern art include a disintegrated bubble chamber window. Two pieces of this glass have been specially decorated by Mario Bellettieri, under the guidance of Eliane de Modzelewska, as mementos for retiring Council members J.H. Bannier and W. Gentner.

>>> Opening record 5/100: https://cds.cern.ch/record/2932376?ln=en
   Found description divs: 2
   DESCRIPTION: From 20 to 27 September the CERN E

### CSV

In [60]:
import csv

output_rows = []


picture_row_index = 0  

# Build rows for CSV — skip only records that have NO picture/link
for i in range(len(titles)):   # normally 0..99

    title = titles[i]
    link = links[i]
    record_url = record_urls[i] if i < len(record_urls) else None
    image_url = image_links[i] if i < len(image_links) else None
    img_filename = image_filenames[i] if i < len(image_filenames) else None
    date = dates[i] if i < len(dates) else None
    tirage = tirages[i] if i < len(tirages) else None
    description = descriptions[i] if i < len(descriptions) else None

    # --- Skip records without a picture ---
    if not link or not link.strip() or not image_url:
        print(f"Skipping record {i+1}: no picture")
        continue

    # Get website row number (only for picture rows)
    website_row_number = row_numbers[picture_row_index]
    picture_row_index += 1

    # Add all columns
    output_rows.append([
        title,
        record_url,
        image_url,            # NEW
        img_filename,
        website_row_number,   # NEW
        date,
        tirage,
        description
    ])

# Write CSV (no index column)
csv_filename = "cern_photolab_records.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([
        "Title",
        "Record URL",
        "Image URL",
        "Image File Name",
        "Row Number on Website",
        "Date",
        "Tirage",
        "Description"
    ])
    writer.writerows(output_rows)

print(f"CSV created: {csv_filename}")
print(f"Total rows written: {len(output_rows)}")


Skipping record 52: no picture
CSV created: cern_photolab_records.csv
Total rows written: 99
